# Sprint 4 — Leakage measurement (the evaluate gate)

**Non-destructive.** Loads the persisted split from `data/processed/`; `02` and its split are untouched (kept as a study reference).

The baseline tree scored **0.9988** on test. But the data holds ~308k duplicate flows (Section 7 #8), and a random split scatters copies of the same flow into *both* train and test. A test row with a twin in train isn't truly unseen — the tree just recalls it. This notebook measures **how much of that 0.9988 is memory, not skill.** The fix (a Section 7 #8 revisit) is the *next* decision, not done here.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

proc = Path('../data/processed')
X_train = pd.read_parquet(proc / 'X_train.parquet')
X_test  = pd.read_parquet(proc / 'X_test.parquet')
y_train = pd.read_parquet(proc / 'y_train.parquet')['label_binary']
y_test  = pd.read_parquet(proc / 'y_test.parquet')['label_binary']

print('train:', X_train.shape, '| test:', X_test.shape)
print('malicious frac  ->  train: %.4f   test: %.4f' % (y_train.mean(), y_test.mean()))

train: (2264502, 65) | test: (566126, 65)
malicious frac  ->  train: 0.1970   test: 0.1970


## 1. Count the twins

The tree only ever sees **features**, so a test row is "leaked" if its exact 65-feature vector also appears in train. We hash each row's features and check the test hashes against the set of train hashes.

Then we split the leaked rows the way we framed them:
- **identical** — same features **and** same label in train → the tree recalls them *correctly* (these inflate the score)
- **fraternal** — same features but only the *other* label in train → the tree recalls them *wrongly*
- **truly unseen** — no feature-twin in train → the only rows that honestly test generalisation

In [2]:
# a test row is "leaked" if its 65-feature vector also appears in train
train_feat = pd.util.hash_pandas_object(X_train, index=False)
test_feat  = pd.util.hash_pandas_object(X_test,  index=False)
leaked = test_feat.isin(set(train_feat))

# split leaked: identical (features + label match) vs fraternal (features match, label never does)
train_fl = pd.util.hash_pandas_object(X_train.assign(_y=y_train.to_numpy()), index=False)
test_fl  = pd.util.hash_pandas_object(X_test.assign(_y=y_test.to_numpy()),  index=False)
identical = test_fl.isin(set(train_fl))
fraternal = leaked & ~identical

print('leaked test rows (feature-twin in train): %7d  (%.2f%% of test)' % (leaked.sum(), 100 * leaked.mean()))
print('   identical (features + SAME label)    : %7d' % identical.sum())
print('   fraternal (features, OTHER label)    : %7d' % fraternal.sum())
print('truly unseen (no twin in train)         : %7d  (%.2f%% of test)' % ((~leaked).sum(), 100 * (~leaked).mean()))

leaked test rows (feature-twin in train):   76235  (13.47% of test)
   identical (features + SAME label)    :   76059
   fraternal (features, OTHER label)    :     176
truly unseen (no twin in train)         :  489891  (86.53% of test)


## 2. The honest score

Refit the exact baseline tree (same split, same seed → reproduces 03's **0.9988**), then score it three ways: overall, on the **leaked** rows (memorised), and on the **clean / truly-unseen** rows. That last line is the honest estimate of how it handles traffic it has never seen — the number that actually matters for an IDS.

In [3]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, recall_score, precision_score

tree = DecisionTreeClassifier(criterion='entropy', max_depth=None, random_state=42).fit(X_train, y_train)
y_pred = pd.Series(tree.predict(X_test), index=X_test.index)

def score(mask, name):
    yt, yp = y_test[mask], y_pred[mask]
    print('%-13s n=%7d   acc=%.4f   recall=%.4f   precision=%.4f' % (
        name, int(mask.sum()), accuracy_score(yt, yp),
        recall_score(yt, yp, zero_division=0), precision_score(yt, yp, zero_division=0)))

print('the reported number:')
score(pd.Series(True, index=X_test.index), 'all test')
print('\nsplit by leakage:')
score(leaked,  'leaked')     # memorised -> should look near-perfect
score(~leaked, 'clean')      # truly unseen -> THE HONEST NUMBER

the reported number:
all test      n= 566126   acc=0.9988   recall=0.9966   precision=0.9972

split by leakage:
leaked        n=  76235   acc=0.9969   recall=0.9959   precision=0.9967
clean         n= 489891   acc=0.9991   recall=0.9969   precision=0.9974
